# 06 - Audit atribut terproteksi pada model fraud (§6.4)
Proposal §6.4: *'no protected attribute in the model'*; fairness diperiksa per wilayah, gender, dan kelompok usia.
Model fraud juara (`XGB mask`, notebook 02b/02c) dilatih dengan **gender, usia (age_ord), dan wilayah (region)** di antara 68 fiturnya. Ini bertentangan dengan proposal.

Notebook ini mengukur **seberapa besar biaya** menghapus ketiganya (dan sekaligus pekerjaan/segmen sebagai pembanding), memakai resep dan protokol yang sama dengan 02b, **hanya pada data DEV**.
Data TEST tidak disentuh; bila keputusan diambil untuk melatih ulang tanpa atribut terproteksi, TEST baru perlu dibuat.

Langkah: (1) latih XGB `mask` dengan set fitur penuh dan tanpa atribut terproteksi, seed sama; (2) bandingkan PR-AUC DEV-OOT pada kondisi bersih dan 20% hilang, plus stabilitas seed; (3) fairness per kelompok usia, gender, wilayah (recall dan laju alert).

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "10")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd()); import splits as S
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
dev = S.load_dev()
for c in S.CAT: dev[c] = dev[c].astype("category")
pool, oot = S.cv_pool(dev), S.dev_oot(dev); L, W = S.LABEL, S.WEIGHT
FEATS = S.features(dev); PROT = ["gender", "age_ord", "region"]
P = dict(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, min_child_weight=5.0, reg_lambda=1.0, gamma=0.0)
def mask_aug(df, cols, seed):
    r = np.random.default_rng(seed); d = df.copy(); rate = r.uniform(0, 0.4, len(d))
    for c in cols: d[c] = d[c].mask(r.random(len(d)) < rate)
    return pd.concat([df, d], ignore_index=True)
def fit(cols, seed=42):
    tr = mask_aug(pool[cols + [L]], cols, 2000 + seed)
    return XGBClassifier(tree_method="hist", enable_categorical=True, n_jobs=10, random_state=seed, **P).fit(tr[cols], tr[L])
def pr(m, cols, d, rate=0.0, seed=7):
    X = d[cols].copy()
    if rate: 
        r = np.random.default_rng(seed)
        for c in cols: X[c] = X[c].mask(r.random(len(X)) < rate)
    return average_precision_score(d[L], m.predict_proba(X)[:, 1], sample_weight=d[W])
print(f"pool {len(pool):,} | OOT {len(oot):,} | fitur penuh {len(FEATS)} | terproteksi: {PROT}")

pool 359,008 | OOT 129,799 | fitur penuh 68 | terproteksi: ['gender', 'age_ord', 'region']


In [2]:
sets = {"penuh (68 fitur, model juara)": FEATS, "tanpa gender, usia, wilayah (65)": [c for c in FEATS if c not in PROT],
        "tanpa terproteksi + pekerjaan + segmen (63)": [c for c in FEATS if c not in PROT + ["occupation", "segment"]]}
models, rows = {}, {}
for name, cols in sets.items():
    seeds = [fit(cols, s) for s in (42, 1, 2)]; models[name] = (seeds[0], cols)
    clean = [pr(m, cols, oot) for m in seeds]; miss = [pr(m, cols, oot, 0.2) for m in seeds]
    rows[name] = {"PR-AUC bersih (seed 42)": clean[0], "bersih rata-rata 3 seed": np.mean(clean), "std seed": np.std(clean), "20% hilang rata-rata": np.mean(miss), "std seed (20% hilang)": np.std(miss)}
audit = pd.DataFrame(rows).T; audit.round(4)

,PR-AUC bersih (seed 42),bersih rata-rata 3 seed,std seed,20% hilang rata-rata,std seed (20% hilang)
"penuh (68 fitur, model juara)",0.6538,0.6607,0.0146,0.4856,0.0216
"tanpa gender, usia, wilayah (65)",0.6710,0.6619,0.0109,0.4717,0.0071
tanpa terproteksi + pekerjaan + segmen (63),0.6640,0.6615,0.0098,0.4764,0.0074


**Cara membaca:** bila selisih antar baris lebih kecil daripada simpangan antar seed (kolom `std seed`), menghapus atribut terproteksi **tidak mengubah kemampuan model secara terukur**, dan keputusan yang tepat adalah menghapusnya demi kepatuhan.

In [3]:
# fairness DEV-OOT pada ambang alert 0,1% traffic: model penuh vs tanpa atribut terproteksi
def flag(m, cols):
    s = m.predict_proba(oot[cols])[:, 1]; o = np.argsort(-s); w = oot[W].values
    thr = s[o][np.searchsorted(np.cumsum(w[o]), 0.001 * w.sum())]; return s >= thr
out = []
for name in list(sets)[:2]:
    m, cols = models[name]; f = flag(m, cols)
    for col in ("gender", "age_ord", "region"):
        for v, g in oot.assign(f=f).groupby(col):
            gp = g[g[L] == 1]; out.append(dict(model=name.split(" (")[0], group=f"{col}={v}", flag_rate=np.average(g.f, weights=g[W]), recall=gp.f.mean() if len(gp) else np.nan, n_fraud_days=len(gp)))
fair = pd.DataFrame(out)
rr = fair[fair.n_fraud_days >= 30].assign(attr=lambda d: d.group.str.split("=").str[0]).groupby(["model", "attr"]).recall.agg(lambda s: s.max() / s.min()).unstack().round(2)
print("Rasio recall terbesar/terkecil per atribut (kelompok dengan >= 30 hari fraud):"); display(rr)
fair.pivot_table(index="group", columns="model", values=["recall", "flag_rate"]).round(4)

Rasio recall terbesar/terkecil per atribut (kelompok dengan >= 30 hari fraud):


attr,age_ord,gender,region
model,,,
penuh,1.07,1.03,1.28
"tanpa gender, usia, wilayah",1.11,1.01,1.26


flag_rate                              recall  \
model                       penuh tanpa gender, usia, wilayah   penuh   
group                                                                   
age_ord=0                  0.0011                      0.0011  0.8428   
age_ord=1                  0.0011                      0.0011  0.8378   
age_ord=2                  0.0009                      0.0009  0.8756   
age_ord=3                  0.0008                      0.0009  0.8757   
age_ord=4                  0.0014                      0.0012  0.8175   
gender=F                   0.0010                      0.0010  0.8388   
gender=M                   0.0010                      0.0010  0.8606   
region=Bali                0.0004                      0.0007  0.9655   
region=Banten              0.0010                      0.0012  0.8864   
region=DI Yogyakarta       0.0015                      0.0020  0.8750   
region=DKI Jakarta         0.0012                      0.0009  0.8421   
region=Jawa Barat          0.0013                      0.0011  0.8226   
region=Jawa Tengah         0.0009                      0.0007  0.8583   
region=Jawa Timur          0.0008                      0.0010  0.9194   
region=Kalimantan Timur    0.0016                      0.0014  0.9667   
region=Lampung             0.0012                      0.0010  0.7538   
region=Sulawesi Selatan    0.0007                      0.0012  0.8085   
region=Sumatera Selatan    0.0011                      0.0010  0.6000   
region=Sumatera Utara      0.0004                      0.0006  0.8909   

                                                     
model                   tanpa gender, usia, wilayah  
group                                                
age_ord=0                                    0.8428  
age_ord=1                                    0.8378  
age_ord=2                                    0.8860  
age_ord=3                                    0.8811  
age_ord=4                                    0.8016  
gender=F                                     0.8445  
gender=M                                     0.8569  
region=Bali                                  0.9655  
region=Banten                                0.8750  
region=DI Yogyakarta                         0.9000  
region=DKI Jakarta                           0.8421  
region=Jawa Barat                            0.8415  
region=Jawa Tengah                           0.8504  
region=Jawa Timur                            0.8710  
region=Kalimantan Timur                      0.9667  
region=Lampung                               0.7692  
region=Sulawesi Selatan                      0.8298  
region=Sumatera Selatan                      0.6400  
region=Sumatera Utara                        0.8909

## Kesimpulan
Lihat tabel 'audit' dan rasio recall di atas. Bila biaya akurasi tidak melebihi simpangan antar seed dan rasio recall tetap sehat, **rekomendasi: latih ulang model fraud tanpa gender, usia, dan wilayah** (resep `mask`, hyperparameter bawaan), buat bundel `v1.1`, dan evaluasi pada data TEST baru (seed berbeda), karena TEST lama sudah dipakai untuk model v1.